In [13]:
import os
import glob
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF],
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

raw_dir = "../data/raw/"
processed_dir = "../data/processed/"
image_dir = os.path.join(processed_dir, "images")
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(image_dir, exist_ok=True)

pdf_files = glob.glob(os.path.join(raw_dir, "*.pdf"))

In [14]:
for pdf_path in pdf_files:
    file_name = os.path.basename(pdf_path)
    base_name = os.path.splitext(file_name)[0]
    
    doc = converter.convert(pdf_path).document
    
    md_path = os.path.join(processed_dir, f"{base_name}.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(doc.export_to_markdown(image_mode="placeholder"))

[INFO] 2026-08-29 21:31:46,162 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-29 21:31:46,163 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-29 21:31:46,171 [RapidOCR] download_file.py:60: File exists and is valid: C:\Ki_OJT\scientific-research-copilot\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-29 21:31:46,172 [RapidOCR] main.py:50: Using C:\Ki_OJT\scientific-research-copilot\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-29 21:31:46,256 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-29 21:31:46,256 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-29 21:31:46,258 [RapidOCR] download_file.py:60: File exists and is valid: C:\Ki_OJT\scientific-research-copilot\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-29 21:31:46,258 [RapidOCR] main.py:50: Using C:\Ki_OJT\scientific-research-copilot\.venv\Lib\site-packages\rapido

In [15]:
for pdf_path in pdf_files:
    file_name = os.path.basename(pdf_path)
    base_name = os.path.splitext(file_name)[0]

    doc = converter.convert(pdf_path).document

    count = 0 
    for item, _ in doc.iterate_items():
        if item.label == "picture":
            img = item.get_image(doc)
            img_path = os.path.join(image_dir, f"{base_name}_figure_{count}.png")
            img.save(img_path)
            count += 1 

In [18]:
import glob
import json
import ollama

images = glob.glob("../data/processed/images/*.png")
results = []

for img_path in images:

    paper_id = img_path.replace("\\", "/").split("/")[-1].split("_")[0]
    prompt = f"""You are a scientific researcher. This is a figure from paper {paper_id}. Describe the image accurately, include:
        1. The type of visual (architecture diagram, chart, graph, photograph, table, etc.).
        2. The main purpose of the figure.
        3. Important components, entities, and relationships.
        4. All important labels and visible text.
        5. Key numerical values, trends, or comparisons if present.
        6. Any conclusions that can reasonably be inferred.
        Do not hallucinate information that is not visible.
        Return a concise but information-dense description.Describe its charts, axes, text, and architecture flow in detail for a search database."""
    
    response = ollama.chat(
        model="llava",
        messages=[{
            "role": "user",
            "content": prompt,
            "images": [img_path]
        }]
    )
    
    results.append({
        "image_id": img_path.replace("\\", "/").split("/")[-1],
        "paper_id": paper_id,
        "description": response['message']['content']
    })

with open("../data/processed/image_descriptions.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)